In [1]:
!pip install tensorflow

In [2]:
pip install scikeras

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
import warnings

# Suppress specific warnings from scikeras
warnings.filterwarnings("ignore", category=UserWarning, module="scikeras")
warnings.filterwarnings("ignore", category=FutureWarning, module="scikeras")
warnings.filterwarnings("ignore", category=UserWarning, message="`model.predict_proba` is deprecated")


In [3]:
df = pd.read_csv('Alphabets_data.csv')

In [4]:
df.head()

,letter,xbox,ybox,width,height,onpix,xbar,ybar,x2bar,y2bar,xybar,x2ybar,xy2bar,xedge,xedgey,yedge,yedgex
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   letter  20000 non-null  object
 1   xbox    20000 non-null  int64 
 2   ybox    20000 non-null  int64 
 3   width   20000 non-null  int64 
 4   height  20000 non-null  int64 
 5   onpix   20000 non-null  int64 
 6   xbar    20000 non-null  int64 
 7   ybar    20000 non-null  int64 
 8   x2bar   20000 non-null  int64 
 9   y2bar   20000 non-null  int64 
 10  xybar   20000 non-null  int64 
 11  x2ybar  20000 non-null  int64 
 12  xy2bar  20000 non-null  int64 
 13  xedge   20000 non-null  int64 
 14  xedgey  20000 non-null  int64 
 15  yedge   20000 non-null  int64 
 16  yedgex  20000 non-null  int64 
dtypes: int64(16), object(1)
memory usage: 2.6+ MB


In [6]:
df.describe()

,xbox,ybox,width,height,onpix,xbar,ybar,x2bar,y2bar,xybar,x2ybar,xy2bar,xedge,xedgey,yedge,yedgex
count,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000,20000.000000,20000.000000,20000.000000,20000.00000
mean,4.023550,7.035500,5.121850,5.37245,3.505850,6.897600,7.500450,4.628600,5.178650,8.282050,6.45400,7.929000,3.046100,8.338850,3.691750,7.80120
std,1.913212,3.304555,2.014573,2.26139,2.190458,2.026035,2.325354,2.699968,2.380823,2.488475,2.63107,2.080619,2.332541,1.546722,2.567073,1.61747
min,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,3.000000,5.000000,4.000000,4.00000,2.000000,6.000000,6.000000,3.000000,4.000000,7.000000,5.00000,7.000000,1.000000,8.000000,2.000000,7.00000
50%,4.000000,7.000000,5.000000,6.00000,3.000000,7.000000,7.000000,4.000000,5.000000,8.000000,6.00000,8.000000,3.000000,8.000000,3.000000,8.00000
75%,5.000000,9.000000,6.000000,7.00000,5.000000,8.000000,9.000000,6.000000,7.000000,10.000000,8.00000,9.000000,4.000000,9.000000,5.000000,9.00000
max,15.000000,15.000000,15.000000,15.00000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.00000,15.000000,15.000000,15.000000,15.000000,15.00000


In [7]:
X = df.iloc[:, :-1] # All columns except the last one as features
y = df.iloc[:, -1]  # The last column as the target

In [8]:
print(f"\nNumber of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Original target classes: {y.unique()}")
print(f"Number of classes: {len(y.unique())}")


Number of samples: 20000
Number of features: 16
Original target classes: [ 8 10  9  7  6 11  4  5  3 12 13 14  1  2 15  0]
Number of classes: 16


In [9]:
# Encode target variable (alphabets to numerical labels)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"\nEncoded target classes (first 5): {y_encoded[:5]}")
print(f"Mapping of classes: {list(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


Encoded target classes (first 5): [ 8 10  9  8 10]
Mapping of classes: [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (13, 13), (14, 14), (15, 15)]


In [10]:
X = X.apply(pd.to_numeric, errors='coerce')

In [11]:
# Fill any NaN values (original or introduced by coerce) with the mean of their respective columns
print("Filling any NaN values in features with column means...")
X = X.fillna(X.mean())
print("NaN values in features filled.")
print("\nMissing values in features (after handling):")
print(X.isnull().sum())
X.isnull().sum().sum()

Filling any NaN values in features with column means...
NaN values in features filled.

Missing values in features (after handling):
letter    20000
xbox          0
ybox          0
width         0
height        0
onpix         0
xbar          0
ybar          0
x2bar         0
y2bar         0
xybar         0
x2ybar        0
xy2bar        0
xedge         0
xedgey        0
yedge         0
dtype: int64


20000

In [12]:
all_nan_cols_after_coerce = X.columns[X.isnull().all()]
if not all_nan_cols_after_coerce.empty: # Conditional print for user feedback
    print(f"\nWarning: The following columns became entirely NaN after conversion and will be filled with 0: {all_nan_cols_after_coerce.tolist()}")
X[all_nan_cols_after_coerce] = X[all_nan_cols_after_coerce].fillna(0)

# --- NEW: Handle columns with zero variance BEFORE scaling ---
# StandardScaler will produce NaNs/Infs if it tries to scale a column with zero standard deviation.
print("\nChecking for and handling zero-variance columns...")
zero_variance_cols = X.columns[X.std() == 0].tolist()
if zero_variance_cols:
    print(f"Warning: The following columns have zero variance and will be dropped: {zero_variance_cols}")
    X = X.drop(columns=zero_variance_cols)
    print(f"Dropped {len(zero_variance_cols)} zero-variance columns.")
else:
    print("No zero-variance columns found.")


# Final critical checks for data integrity before proceeding (these use if-else for halting execution)
if X.isnull().sum().sum() > 0:
    print("\nCRITICAL ERROR: NaN values still present in X after all preprocessing steps. Exiting.")
    exit()
if np.isinf(X).any().any():
    print("\nCRITICAL ERROR: Infinite values still present in X after all preprocessing steps. Exiting.")
    exit()
print("\nSuccessfully handled all NaN and infinite values in features (X).")





Checking for and handling zero-variance columns...
Dropped 1 zero-variance columns.

Successfully handled all NaN and infinite values in features (X).


In [13]:
# Encode target variable (alphabets to numerical labels)
# Ensure y is treated as string before encoding to avoid issues if it's mixed type
y = y.astype(str)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"\nEncoded target classes (first 5): {y_encoded[:5]}")
print(f"Mapping of classes: {list(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


Encoded target classes (first 5): [14  2 15 14  2]
Mapping of classes: [('0', 0), ('1', 1), ('10', 2), ('11', 3), ('12', 4), ('13', 5), ('14', 6), ('15', 7), ('2', 8), ('3', 9), ('4', 10), ('5', 11), ('6', 12), ('7', 13), ('8', 14), ('9', 15)]


In [14]:
# Data Normalization (StandardScaler is generally good for ANNs)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nFeatures scaled successfully using StandardScaler.")
print(f"Scaled features shape: {X_scaled.shape}")


Features scaled successfully using StandardScaler.
Scaled features shape: (20000, 15)


In [15]:
# --- 2. Model Implementation ---
print("\n--- 2. Model Implementation ---")



--- 2. Model Implementation ---


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")


Training set shape: (16000, 15)
Test set shape: (4000, 15)


In [17]:
# --- FINAL DIAGNOSTIC CHECKS ON X_TRAIN BEFORE MODEL TRAINING ---
print("\n--- Final checks on X_train before model training ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_train data type: {X_train.dtype}")
print(f"Number of NaN values in X_train: {np.isnan(X_train).sum()}")
print(f"Number of infinite values in X_train: {np.isinf(X_train).sum()}")
print(f"Min value in X_train: {np.min(X_train)}")
print(f"Max value in X_train: {np.max(X_train)}")
print("Descriptive statistics for X_train (first 5 columns):")
print(pd.DataFrame(X_train).iloc[:, :5].describe())


--- Final checks on X_train before model training ---
X_train shape: (16000, 15)
X_train data type: float64
Number of NaN values in X_train: 0
Number of infinite values in X_train: 0
Min value in X_train: -4.744893657580275
Max value in X_train: 5.73732917798792
Descriptive statistics for X_train (first 5 columns):
                  0             1             2             3             4
count  16000.000000  16000.000000  16000.000000  16000.000000  16000.000000
mean       0.005495      0.006147      0.005411      0.008203      0.005718
std        0.999393      0.997867      0.996198      0.995608      0.997362
min       -2.103087     -2.129084     -2.542463     -2.375788     -1.600550
25%       -0.535004     -0.615983     -0.556881     -0.606921     -0.687476
50%       -0.012309     -0.010743     -0.060486      0.277513     -0.230939
75%        0.510385      0.594497      0.435910      0.719730      0.682135
max        5.737329      2.410218      4.903469      4.257465      5.24750

In [18]:
# Final critical checks for data integrity in X_train before model training
if np.isnan(X_train).any():
    print("\nCRITICAL ERROR: NaN values still found in X_train BEFORE model training. This should not happen if previous steps worked. Exiting.")
    exit()
if np.isinf(X_train).any():
    print("\nCRITICAL ERROR: Infinite values still found in X_train BEFORE model training. Exiting.")
    exit()
print("\nNo NaN or infinite values found in X_train. Proceeding with model training.")



No NaN or infinite values found in X_train. Proceeding with model training.


In [19]:
# Function to create the Keras model (will be used for both basic and tuning)
def create_model(num_hidden_layers=1, neurons_per_layer=64, activation='relu', learning_rate=0.001):
    model = Sequential()
    # Input layer
    model.add(Dense(neurons_per_layer, input_dim=X_train.shape[1], activation=activation))

    # Hidden layers
    for _ in range(num_hidden_layers - 1): # -1 because the first hidden layer is already added
        model.add(Dense(neurons_per_layer, activation=activation))

    # Output layer
    # Use 'softmax' for multi-class classification and num_classes for output units
    model.add(Dense(num_classes, activation='softmax'))

    # Compile the model
    optimizer = Adam(learning_rate=learning_rate)
    # Use 'sparse_categorical_crossentropy' when target labels are integers (not one-hot encoded)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [20]:
# --- Basic ANN Model ---
print("\n--- Training Basic ANN Model (Default Hyperparameters) ---")
# Using default parameters as defined in create_model function
basic_model = create_model()
basic_model.summary()


--- Training Basic ANN Model (Default Hyperparameters) ---


C:\Users\Rajiv Anatwar\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,064 (8.06 KB)

 Trainable params: 2,064 (8.06 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history_basic = basic_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)
print("Basic model training complete.")

Basic model training complete.


In [22]:
# Evaluate the basic model
loss_basic, accuracy_basic = basic_model.evaluate(X_test, y_test, verbose=0)
print(f"\nBasic Model Test Accuracy: {accuracy_basic:.4f}")
print(f"Basic Model Test Loss: {loss_basic:.4f}")


Basic Model Test Accuracy: 0.6053
Basic Model Test Loss: 1.0510


In [23]:
y_pred_basic = np.argmax(basic_model.predict(X_test), axis=1)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [24]:
print("\nBasic Model Classification Report:")
print(classification_report(y_test, y_pred_basic,
                            target_names=label_encoder.classes_,
                            labels=np.arange(num_classes), # Explicitly provide all possible labels
                            zero_division=0))



Basic Model Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.50      0.67      0.57         3
          10       0.46      0.51      0.48       316
          11       0.66      0.60      0.63       174
          12       0.40      0.07      0.12        27
          13       0.17      0.10      0.12        10
          14       0.00      0.00      0.00         3
          15       0.00      0.00      0.00         0
           2       0.50      0.17      0.25         6
           3       0.54      0.27      0.36        26
           4       0.59      0.46      0.51        96
           5       0.50      0.35      0.41       198
           6       0.55      0.42      0.48       365
           7       0.49      0.55      0.52       694
           8       0.74      0.81      0.77      1610
           9       0.46      0.40      0.43       472

    accuracy                           0.61 

In [40]:
# --- 3. Hyperparameter Tuning ---
print("\n--- 3. Hyperparameter Tuning using RandomizedSearchCV ---")


--- 3. Hyperparameter Tuning using RandomizedSearchCV ---


In [26]:
# Create a KerasClassifier wrapper for scikit-learn's GridSearchCV
# Pass build_fn as the function to create the Keras model
model_for_tuning = KerasClassifier(model=create_model, verbose=0, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [31]:
# Define the hyperparameter grid to search
param_dist = { # Changed from param_grid to param_dist
    'model__num_hidden_layers': [1, 2],
    'model__neurons_per_layer': [64, 128],
    'model__activation': ['relu', 'tanh'],
    'model__learning_rate': [0.001, 0.01],
    'batch_size': [32, 64],
    'epochs': [10, 20, 30] # Reduced max epochs for faster tuning
   
    
}

In [32]:
# Initialize GridSearchCV
# cv=3 means 3-fold cross-validation
# n_jobs=-1 uses all available CPU cores
random_search = RandomizedSearchCV(estimator=model_for_tuning,
                                   param_distributions=param_dist, # Changed from param_grid
                                   n_iter=10, # Try 10 random combinations (adjust as needed)
                                   cv=3,
                                   scoring='accuracy',
                                   verbose=1,
                                   n_jobs=-1,
                                   random_state=42, # For reproducibility
                                   error_score='raise')


In [33]:
print("\nStarting Randomized Search... This will be much faster than Grid Search.")
grid_result = random_search.fit(X_train, y_train) # Renamed variable for consistency with previous output
print("Randomized Search complete.")


Starting Randomized Search... This will be much faster than Grid Search.
Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\Rajiv Anatwar\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
C:\Users\Rajiv Anatwar\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Randomized Search complete.


In [34]:
print(f"\nBest Score: {grid_result.best_score_:.4f} using {grid_result.best_params_}")


Best Score: 0.6256 using {'model__num_hidden_layers': 2, 'model__neurons_per_layer': 64, 'model__learning_rate': 0.01, 'model__activation': 'tanh', 'epochs': 20, 'batch_size': 64}


In [36]:
best_model = grid_result.best_estimator_

In [41]:
loss_tuned, accuracy_tuned = best_model.model_.evaluate(X_test, y_test, verbose=0)
print(f"\nTuned Model Test Accuracy: {accuracy_tuned:.4f}")
print(f"Tuned Model Test Loss: {loss_tuned:.4f}")


Tuned Model Test Accuracy: 0.6430
Tuned Model Test Loss: 0.9699


In [43]:
y_pred_tuned = np.argmax(best_model.model_.predict(X_test), axis=1)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [44]:
print("\nTuned Model Classification Report:")
print(classification_report(y_test, y_pred_tuned,
                            target_names=label_encoder.classes_,
                            labels=np.arange(num_classes), # Explicitly provide all possible labels
                            zero_division=0))


Tuned Model Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.33      0.50         3
          10       0.65      0.39      0.49       316
          11       0.59      0.70      0.64       174
          12       0.26      0.19      0.22        27
          13       0.18      0.20      0.19        10
          14       0.00      0.00      0.00         3
          15       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         6
           3       0.29      0.54      0.37        26
           4       0.69      0.34      0.46        96
           5       0.43      0.57      0.49       198
           6       0.53      0.48      0.51       365
           7       0.62      0.52      0.56       694
           8       0.76      0.85      0.80      1610
           9       0.50      0.55      0.53       472

    accuracy                           0.64 

In [45]:
# --- Discussion of Performance Differences ---
print("\n--- Performance Comparison ---")
print(f"Basic Model Accuracy: {accuracy_basic:.4f}")
print(f"Tuned Model Accuracy: {accuracy_tuned:.4f}")


--- Performance Comparison ---
Basic Model Accuracy: 0.6053
Tuned Model Accuracy: 0.6430


In [46]:
if accuracy_tuned > accuracy_basic:
    print("\nObservation: The tuned model achieved higher accuracy compared to the basic model.")
    print("This indicates that hyperparameter tuning effectively improved the model's performance.")
else:
    print("\nObservation: The tuned model's accuracy is not significantly higher or is lower than the basic model.")
    print("This could mean the default parameters were already good, or the search space for tuning needs to be expanded/refined.")


Observation: The tuned model achieved higher accuracy compared to the basic model.
This indicates that hyperparameter tuning effectively improved the model's performance.


In [47]:
print("\nDetailed discussion on the effects of hyperparameter tuning:")
print("- **Number of Hidden Layers and Neurons:** Increasing layers/neurons can capture more complex patterns but risk overfitting.")
print("- **Activation Functions:** 'relu' is common for hidden layers, 'tanh' can also work. 'softmax' is crucial for output in multi-class classification.")
print("- **Learning Rate:** A crucial hyperparameter. Too high, and the model might overshoot the optimal solution; too low, and training might be very slow or get stuck.")
print("- **Batch Size and Epochs:** Affect training dynamics and convergence. Optimal values balance training speed and model generalization.")
print("\nBy systematically searching through different combinations, RandomizedSearchCV helps find a set of hyperparameters that lead to better generalization on unseen data, as demonstrated by the improved (or stable) test accuracy and other metrics.")

print("\n--- ANN Classification with Hyperparameter Tuning Complete ---")


Detailed discussion on the effects of hyperparameter tuning:
- **Number of Hidden Layers and Neurons:** Increasing layers/neurons can capture more complex patterns but risk overfitting.
- **Activation Functions:** 'relu' is common for hidden layers, 'tanh' can also work. 'softmax' is crucial for output in multi-class classification.
- **Learning Rate:** A crucial hyperparameter. Too high, and the model might overshoot the optimal solution; too low, and training might be very slow or get stuck.
- **Batch Size and Epochs:** Affect training dynamics and convergence. Optimal values balance training speed and model generalization.

By systematically searching through different combinations, RandomizedSearchCV helps find a set of hyperparameters that lead to better generalization on unseen data, as demonstrated by the improved (or stable) test accuracy and other metrics.

--- ANN Classification with Hyperparameter Tuning Complete ---


Evaluation and Discussion of ANN Model Performance
This report summarizes the evaluation of the Artificial Neural Network (ANN) classification model on the "Alphabets_data.csv" dataset, comparing a basic model against
a hyperparameter-tuned model.

1. Model Performance Overview
   
Both models were evaluated using accuracy, precision, recall, and F1-score.

Basic Model Performance
Test Accuracy: (Value from Python output)

Test Loss: (Value from Python output)


Tuned Model Performance

The hyperparameter-tuned model was optimized using RandomizedSearchCV.

Best Hyperparameters Found: (Values from grid_result.best_params_ in Python output)

model__num_hidden_layers: [...]

model__neurons_per_layer: [...]

model__activation: [...]

model__learning_rate: [...]

batch_size: [...]

epochs: [...]

Best Cross-Validation Score: (Value from Python output)

Test Accuracy: (Value from Python output)

Test Loss: (Value from Python output

2. Discussion of Performance Differences
   
Overall Accuracy: The tuned model achieved a [state whether higher, lower, or similar, and by how much] test accuracy (Tuned: [value] vs. Basic: [value]). This indicates [interpret the change, e.g., "a significant improvement," "a modest gain," or "no substantial change"] in generalization.

Per-Class Metrics: A detailed look at the classification reports reveals that the tuned model generally improved [mention specific metrics like precision, recall, or F1-score] for [mention if specific classes or overall] alphabet categories, suggesting better discrimination or reduced errors for these.

Effects of Hyperparameter Tuning
Hyperparameter tuning directly influenced the model's ability to learn effectively:

Network Architecture (Layers/Neurons): Optimizing the number of hidden layers and neurons allowed the model to capture more appropriate data complexities without overfitting.

Activation Functions: The choice of activation (e.g., 'relu') enabled efficient introduction of non-linearity, crucial for learning complex patterns in the alphabet data.

Learning Rate: A well-chosen learning rate facilitated smoother and faster convergence during training.

Batch Size & Epochs: Tuning these parameters helped balance training efficiency and the model's capacity to generalize well to unseen data.

In summary, the systematic tuning process, even with RandomizedSearchCV for efficiency, led to a [reiterate the overall outcome, e.g., "more accurate and robust"] classification model by optimizing its internal learning mechanisms.

3. Conclusion

This assignment successfully demonstrated the impact of hyperparameter tuning on ANN performance for alphabet classification. The tuned model generally outperformed the basic model, highlighting the importance of this optimization step in machine learning workflows.